In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpg7zyutzu".


In [2]:
%%cuda
#include <iostream>

// THE KERNEL
__global__ void reverseArray(float* d_out, float* d_in, int n) {
    int t = blockDim.x * blockIdx.x + threadIdx.x;

    if (t < n) {
        // The Magic Formula:
        // If n = 10, and t = 0 -> index becomes (10 - 1 - 0) = 9
        // If n = 10, and t = 9 -> index becomes (10 - 1 - 9) = 0
        int targetIndex = n - 1 - t;
        d_out[targetIndex] = d_in[t];
    }
}

int main() {
    int n = 8;
    size_t size = n * sizeof(float);

    float *h_in = (float*)malloc(size);
    float *h_out = (float*)malloc(size);

    for(int i=0; i<n; i++) h_in[i] = (float)i; // [0, 1, 2, 3, 4, 5, 6, 7]

    float *d_in, *d_out;
    cudaMalloc(&d_in, size);
    cudaMalloc(&d_out, size);

    cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice);

    // Launch 1 block with 8 threads
    reverseArray<<<1, 8>>>(d_out, d_in, n);

    cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost);

    std::cout << "Index:  0 1 2 3 4 5 6 7" << std::endl;
    std::cout << "In:     ";
    for(int i=0; i<n; i++) std::cout << h_in[i] << " ";
    std::cout << "\nOut:    ";
    for(int i=0; i<n; i++) std::cout << h_out[i] << " ";
    std::cout << std::endl;

    cudaFree(d_in); cudaFree(d_out);
    free(h_in); free(h_out);
    return 0;
}

Index:  0 1 2 3 4 5 6 7
In:     0 1 2 3 4 5 6 7 
Out:    7 6 5 4 3 2 1 0 

